# Oscar winners for the AWARD_Actor table
Written by: Capleton

This notebook makes a clean CSV file that fits the table AWARD_Actor in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## STEP 2: read the file
The original file stays unchanged. If something goes wrong in the database, we can always build the table again from the full data.

In [2]:
df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")

print("Rows and columns:", df.shape)
print(df.head())

Rows and columns: (10889, 7)
   year_film  year_ceremony  ceremony category                 name              film  winner
0       1927           1928         1    ACTOR  Richard Barthelmess         The Noose   False
1       1927           1928         1    ACTOR        Emil Jannings  The Last Command    True
2       1927           1928         1  ACTRESS       Louise Dresser   A Ship Comes In   False
3       1927           1928         1  ACTRESS         Janet Gaynor        7th Heaven    True
4       1927           1928         1  ACTRESS       Gloria Swanson    Sadie Thompson   False


## STEP 3: keep only the winners with a film
The database should only hold data that answers our research questions. An award without a film can never be linked to a row in MOVIE, so it has no place in a table with a foreign key to MOVIE.

In [3]:
winners = df[df["winner"] == True]
winners = winners.dropna(subset=["film"])

print("Rows before:", len(df))
print("Rows after:", len(winners))

Rows before: 10889
Rows after: 2172


## STEP 4: keep only the acting awards
In our ERD every type of award has its own table. AWARD_Actor has a foreign key to ACTOR, so it may only hold Oscars that belong to an actor or an actress. All acting categories start with ACTOR or ACTRESS, for leading and supporting roles.

In [4]:
award_actor = winners[winners["category"].str.startswith(("ACTOR", "ACTRESS"))].copy()

print("Rows after:", len(award_actor))
print("Categories:", list(award_actor["category"].unique()))

Rows after: 370
Categories: ['ACTOR', 'ACTRESS', 'ACTOR IN A SUPPORTING ROLE', 'ACTRESS IN A SUPPORTING ROLE', 'ACTOR IN A LEADING ROLE', 'ACTRESS IN A LEADING ROLE']


## STEP 5: rename the columns
The column names in the CSV must be the same as in the table in PostgreSQL. Otherwise pgAdmin does not know which value goes into which column.

In [5]:
award_actor = award_actor.rename(columns={"name": "name_nominee", "film": "film_title"})

print(list(award_actor.columns))

['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']


## STEP 6: clean the film titles and make the match key
Some old titles in the source data contain a ;. It is a leftover and not part of the real title, so we remove it to keep the data in the database clean.

film_title_clean is the title in small letters, a hyphen becomes a space, dots are removed and double spaces become one. film_title stays as it is, so the real title with capitals is still stored.

match_key is film_title_clean plus the year of the film, for example no country for old men_2007. The database needs it to fill the foreign key movie_id: the Oscar data has no movie_id, so we look up this key in MOVIE. We add the year because different films can have the same title, like a remake. The year of the film is year_ceremony minus 1, because the Oscars are given the year after the film came out. MOVIE makes the same key from its title and the year of its release date.

In [6]:
award_actor["film_title"] = award_actor["film_title"].str.replace(";", "").str.strip()
award_actor["film_title_clean"] = award_actor["film_title"].str.lower().str.replace("-", " ", regex=False).str.replace(".", "", regex=False).str.split().str.join(" ")

award_actor["match_key"] = award_actor["film_title_clean"] + "_" + (award_actor["year_ceremony"] - 1).astype(str)

print(award_actor[["film_title", "film_title_clean", "match_key"]].head())
print("Titles with ; left:", award_actor["film_title"].str.contains(";").sum())

          film_title  film_title_clean              match_key
1   The Last Command  the last command  the last command_1927
3         7th Heaven        7th heaven        7th heaven_1927
36    In Old Arizona    in old arizona    in old arizona_1928
45          Coquette          coquette          coquette_1928
73          Disraeli          disraeli          disraeli_1929
Titles with ; left: 0


## STEP 7: sort the table
A fixed order gives every award the same actor_award_id every time. If the numbers change, the rows in the database no longer match the file.

In [7]:
award_actor = award_actor.sort_values(["year_ceremony", "category", "film_title"]).reset_index(drop=True)

print(award_actor[["year_ceremony", "category", "name_nominee", "film_title", "film_title_clean"]].head())

   year_ceremony category   name_nominee        film_title  film_title_clean
0           1928    ACTOR  Emil Jannings  The Last Command  the last command
1           1928  ACTRESS   Janet Gaynor        7th Heaven        7th heaven
2           1929    ACTOR  Warner Baxter    In Old Arizona    in old arizona
3           1929  ACTRESS  Mary Pickford          Coquette          coquette
4           1930    ACTOR  George Arliss          Disraeli          disraeli


## STEP 8: give every award an actor_award_id
actor_award_id is the primary key of AWARD_Actor. The database needs it to tell every row apart, so it must be unique and can never be empty.

In [8]:
award_actor["actor_award_id"] = award_actor.index + 1

print("First actor_award_id:", award_actor["actor_award_id"].min())
print("Last actor_award_id:", award_actor["actor_award_id"].max())
print("All unique:", award_actor["actor_award_id"].is_unique)

First actor_award_id: 1
Last actor_award_id: 370
All unique: True


## STEP 9: add empty actor_id and movie_id
Both are foreign keys in our ERD, to ACTOR and to MOVIE. The columns must exist to fit the table, but the numbers are not in the Oscar data. actor_id is filled in the Actorclean notebook, movie_id later with the match on title and year.

In [9]:
award_actor["actor_id"] = None
award_actor["movie_id"] = None

print(award_actor.head())

   year_film  year_ceremony  ceremony category   name_nominee        film_title  winner  film_title_clean              match_key  actor_award_id actor_id movie_id
0       1927           1928         1    ACTOR  Emil Jannings  The Last Command    True  the last command  the last command_1927               1     None     None
1       1927           1928         1  ACTRESS   Janet Gaynor        7th Heaven    True        7th heaven        7th heaven_1927               2     None     None
2       1928           1929         2    ACTOR  Warner Baxter    In Old Arizona    True    in old arizona    in old arizona_1928               3     None     None
3       1928           1929         2  ACTRESS  Mary Pickford          Coquette    True          coquette          coquette_1928               4     None     None
4       1929           1930         3    ACTOR  George Arliss          Disraeli    True          disraeli          disraeli_1929               5     None     None


## STEP 10: keep only the ERD columns
The CSV must have exactly the same columns as the table in PostgreSQL. The primary key comes first, then the foreign keys, so the most important columns are easy to find. winner is removed because every row is a winner, and year_film because it is always year_ceremony minus 1. Storing the same information twice goes against normalization. film_title_clean and match_key are not in the ERD. They stay in the file for now, because they are needed to fill movie_id. They are removed when movie_id is filled, before the import in pgAdmin.

In [10]:
award_actor = award_actor[["actor_award_id", "actor_id", "movie_id", "year_ceremony",
                                 "ceremony", "category", "name_nominee", "film_title", "film_title_clean", "match_key"]]

print(list(award_actor.columns))

['actor_award_id', 'actor_id', 'movie_id', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'film_title_clean', 'match_key']


## STEP 11: save as CSV
This file is imported into AWARD_Actor in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table and the import would fail.

In [11]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_actor_clean.csv")

award_actor.to_csv(new_file, index=False)

print("Saved:", new_file)
print("File exists:", os.path.exists(new_file))

Saved: C:\Users\caple\OneDrive - HvA\Master\Data management\Data\award_actor_clean.csv
File exists: True
